# scMOBA-Chat: human M1 single-cell inference


Use the scMOBA-Chat environment, an authorized M1 `.h5ad`, and the full `chat/` asset directory from the model Hub. This example assumes human Ensembl IDs in `var_names` and already log1p-normalized expression in `X`; adjust the arguments to match your file.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess, sys
import anndata as ad
from datasets import load_from_disk
from scmoba_chat.inference import Engine

m1 = Path('data/m1.h5ad').resolve()
config = Path('models/chat/config.json').resolve()
assert m1.is_file() and config.is_file()
work = Path(mkdtemp(prefix='scmoba-chat-m1-'))
source = ad.read_h5ad(m1, backed='r')
try:
    subset = source[:min(16, source.n_obs)].to_memory()
finally:
    source.file.close()
subset.obs = subset.obs.iloc[:, :0].copy()
subset.write_h5ad(work / 'm1-local.h5ad')


In [ ]:
subprocess.run([sys.executable, '-m', 'scmoba_chat.cli', 'prepare',
    '--input', str(work / 'm1-local.h5ad'), '--output', str(work / 'prepared'),
    '--resources', str(config.parent / 'resources.json'), '--input-scale', 'log1p',
    '--gene-type', 'ensembl'], check=True)
cell = load_from_disk(str(work / 'prepared' / 'tokens'))[0]
engine = Engine(config)
print(engine.chat(cell['input_ids'], task='subclass')['answer'])


In [ ]:
first = engine.chat(cell['input_ids'], 'What genes characterize this cell?')
follow = engine.chat(cell['input_ids'], 'Which biological pathways are prominent in this cell?', history=first['history'])
print(first['answer'])
print(follow['answer'])
